### Extração de características dos dados do Benchmark Dataset (Tsinghua Lab) para o participante S1

Este notebook lê as epochs filtradas do participante S1, extrai características no demínio do tempo e da frequência de cada canal selecionado e salva o vetor de características e seus rótulos para a etapa de classificação.

**Características**

Inicialmente, foram usadas as nove características apresentadas em aula. Em que, nas fórmulas, N é o número de amostrar do sinal e M é o número de pontos do espectro.

Domínio do tempo

1. Willison Amplitude (WAMP)

    > $ \sum_{i=1}^{N-1}f(|x_i - x_{i+1}|) $

    > $ f(x) = \begin{cases} 1 & \text{if } x \gt threshold \\ 0 & \text{otherwise} \end{cases} $

2. Variance of EMG (VAR)

    > $ \frac{1}{N-1}\sum_{i=1}^{N}x_i^2 $

3. Root Mean Square (RMS)

    > $ \sqrt{\frac{1}{N}\sum_{i=1}^{N}|x_i|^2} $

4. Waveform Length (WL)

    > $ \sum_{i=1}^{N-1}|x_{i+1} - x_i| $

5. Zero Crossing (ZC)

    > $ \sum_{i=1}^{N-1}sgn(x_i) $

    > $ sgn(x) = \begin{cases} 1 & \text{if } x_i * x_{i+1} \leq 0 \\ 0 & \text{otherwise} \end{cases} $

Domínio da frequência

1. Median Frequency (FMD)

    > $ \frac{1}{2}\sum_{j=1}^{M}PSD_j $

2. Mean Frequency (FMN)

    > $ \sum_{j=1}^{M}f_j PSD_j \Big/ \sum_{j=1}^{M}PSD_j $

    > $ f_j = \frac{j \cdot SampleRate}{2M} $

3. Modified Median Frequency (MMDF)

    > $ \frac{1}{2}\sum_{j=1}^{M}A_j $

    > $ A_j = Amplitude\ do\ espectro\ j $

4. Modified Frequency Mean (MMNF)

    > $ \sum_{j=1}^{M}f_jA_j \Big/ \sum_{j=1}^{M}A_j $

In [1]:
# Carregar as epochs filtradas

import mne
import numpy as np

epochs = mne.read_epochs('data/output/S1_filtered_epo.fif', preload=True, verbose=False)
fs = epochs.info['sfreq']

print(epochs)

<EpochsFIF | 240 events (all good), -0.5 – 5.496 s (baseline off), ~175.8 MiB, data loaded, with metadata,
 '8.0': 6
 '9.0': 6
 '10.0': 6
 '11.0': 6
 '12.0': 6
 '13.0': 6
 '14.0': 6
 '15.0': 6
 '8.2': 6
 '9.2': 6
 and 30 more events ...>


Foram selecionados os nove eletrodos usados na avaliação com FBCCA do artigo referência (Pz, PO5, PO3, POz, PO4, PO6, O1, Oz e O2). Além disso, foi usado um recorte de 0 a 5 segundos de cada epoch, para descartar os 0,5 segundos antes do estímulo. Por fim, o sinal foi convertido de volts para microvolts, que é a unidade utilizada pelo artigo.

In [2]:
p_occipital = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
selected_channels = [name for name in epochs.ch_names if name.upper() in p_occipital]
print(len(selected_channels), selected_channels)

# Seleciona os canais e recorta o período de estímulo
epochs_selected = epochs.copy().pick(selected_channels).crop(tmin=0, tmax=5)

x = epochs_selected.get_data() * 1e6
y = epochs_selected.events[:, 2]

print(x.shape)
print(y.shape)

9 ['PZ', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
(240, 9, 1251)
(240,)
